In [5]:
# ==============================================================================
# CELL 1: IMPORTS & SETUP
# ==============================================================================

import os
import json
import time
import warnings
import numpy as np

import torch
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from numba import njit

warnings.filterwarnings("ignore")

# ---- Device setup (MacBook, 18GB, Apple Silicon MPS) ----
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"[✓] Using device: {DEVICE}")

OUTPUT_DIR = "optimized_layouts_rl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

[✓] Using device: mps


In [6]:
# ==============================================================================
# CELL 2: LOAD CONFIG & BUILD COMPONENT ARRAYS
# ==============================================================================

def load_config(filepath='newobj.json'):
    if not os.path.exists(filepath):
        filepath = os.path.join(os.getcwd(), 'newobj.json')
    if not os.path.exists(filepath):
        raise FileNotFoundError("❌ Unable to locate 'newobj.json' in working directory.")
    with open(filepath, 'r') as f:
        data = json.load(f)
    print(f"[✓] Config loaded: '{filepath}'")
    return data

data = load_config('newobj.json')

# ---- Board parameters ----
panel_width    = float(data['BOARD']['Length (mm)'])
panel_height   = float(data['BOARD']['Breadth (mm)'])
border_spacing = float(data['BOARD']['Border (mm)'])
clearance      = float(data['BOARD']['Clearance (mm)'])

# ---- Flatten FRONT + BACK components, expanding Object qty > 1 ----
raw_components_to_process = []
for comp in data['COMPONENTS'].get('FRONT', []):
    raw_components_to_process.append((comp, False))
for comp in data['COMPONENTS'].get('BACK', []):
    raw_components_to_process.append((comp, True))

components_to_process = []
for comp, is_back in raw_components_to_process:
    qty = int(comp.get('Object qty', 1))
    if qty > 1:
        for q in range(qty):
            cloned = comp.copy()
            cloned['Unique Name'] = f"{comp['Object name']}{q}"
            components_to_process.append((cloned, is_back))
    else:
        cloned = comp.copy()
        cloned['Unique Name'] = comp['Object name']
        components_to_process.append((cloned, is_back))

num_elements = len(components_to_process)

element_names        = []
element_shapes        = []
is_back_side_list     = []
flat_offsets_list      = []
clearance_dirs_list    = []
element_data           = []  # [mass, length, width, insert_diam]

offset_counts  = np.zeros(num_elements, dtype=np.int32)
offset_indices = np.zeros(num_elements, dtype=np.int32)
current_idx = 0

for idx, (component, is_back) in enumerate(components_to_process):
    element_name = component['Unique Name']
    shape = component.get('Shape', 'rectangle')
    element_shapes.append(shape)
    is_back_side_list.append(1 if is_back else 0)

    # Custom face clearances (CF)
    cf_faces = component.get('CF', [])
    cf_len = float(component.get('CFLen (mm)', clearance))
    c_dirs = [clearance, clearance, clearance, clearance]  # [Top, Right, Bottom, Left]
    for face in cf_faces:
        if face == 1: c_dirs[0] = cf_len
        elif face == 2: c_dirs[1] = cf_len
        elif face == 3: c_dirs[2] = cf_len
        elif face == 4: c_dirs[3] = cf_len
    clearance_dirs_list.append(c_dirs)

    if shape == 'rectangle':
        length = float(component['Length (mm)'])
        width  = float(component['Breadth (mm)'])
        insert_diam = float(component['Insert (mm)'])
        insert_qty  = int(component.get('Insert qty', 4))
        half_l, half_w = length / 2.0, width / 2.0

        if insert_qty == 2:
            offsets = [(half_l, 0.0), (-half_l, 0.0)] if length >= width else [(0.0, half_w), (0.0, -half_w)]
        elif insert_qty == 6:
            offsets = [(half_l, half_w), (half_l, -half_w), (-half_l, half_w), (-half_l, -half_w)]
            offsets.extend([(0.0, half_w), (0.0, -half_w)] if length >= width else [(half_l, 0.0), (-half_l, 0.0)])
        else:
            offsets = [(half_l, half_w), (half_l, -half_w), (-half_l, half_w), (-half_l, -half_w)]
    else:
        diameter = float(component['Diameter (mm)'])
        length, width = diameter, diameter
        insert_diam = float(component['Insert (mm)'])
        insert_qty  = int(component.get('Insert qty', 3))
        inner_radius = max(0.0, (diameter / 2.0) - 1.0)
        offsets = [(inner_radius * np.cos(2*np.pi*k/insert_qty), inner_radius * np.sin(2*np.pi*k/insert_qty))
                   for k in range(insert_qty)]

    flat_offsets_list.extend(offsets)
    offset_counts[idx] = len(offsets)
    offset_indices[idx] = current_idx
    current_idx += len(offsets)

    element_data.append([float(component['Weight (kg)']), length, width, insert_diam])
    element_names.append(element_name)

flat_offsets   = np.array(flat_offsets_list, dtype=np.float64)
element_data   = np.array(element_data, dtype=np.float64)
masses         = element_data[:, 0]
lengths        = element_data[:, 1]
widths         = element_data[:, 2]
insert_diams   = element_data[:, 3]
is_back_side   = np.array(is_back_side_list, dtype=np.int32)
clearance_dirs = np.array(clearance_dirs_list, dtype=np.float64)
n = num_elements

# ---- Required pin-to-pin distance matrix (by insert size) ----
req_d_matrix = np.full((n, n), 30.0)
for i in range(n):
    for j in range(n):
        if insert_diams[i] == 4.0 and insert_diams[j] == 4.0:
            req_d_matrix[i, j] = 24.0
        elif insert_diams[i] == 6.0 and insert_diams[j] == 6.0:
            req_d_matrix[i, j] = 36.0

print(f"[✓] {n} components parsed ({int(is_back_side.sum())} back-side, {n - int(is_back_side.sum())} front-side)")
for i, nm in enumerate(element_names):
    print(f"    {i:2d}. {nm:8s} | {element_shapes[i]:9s} | mass={masses[i]:.3f}kg | pins={offset_counts[i]}")

[✓] Config loaded: 'newobj.json'
[✓] 18 components parsed (8 back-side, 10 front-side)
     0. PDS      | rectangle | mass=1.500kg | pins=4
     1. CCRR0    | circle    | mass=0.150kg | pins=6
     2. CCRR1    | circle    | mass=0.150kg | pins=6
     3. LD0      | circle    | mass=0.040kg | pins=3
     4. LD1      | circle    | mass=0.040kg | pins=3
     5. LD2      | circle    | mass=0.040kg | pins=3
     6. LD3      | circle    | mass=0.040kg | pins=3
     7. LD4      | circle    | mass=0.040kg | pins=3
     8. LD5      | circle    | mass=0.040kg | pins=3
     9. RS       | rectangle | mass=1.500kg | pins=4
    10. BM1      | rectangle | mass=1.500kg | pins=4
    11. PB0      | rectangle | mass=0.100kg | pins=4
    12. PB1      | rectangle | mass=0.100kg | pins=4
    13. PB2      | rectangle | mass=0.100kg | pins=4
    14. PB3      | rectangle | mass=0.100kg | pins=4
    15. RPU      | rectangle | mass=3.000kg | pins=6
    16. MES      | rectangle | mass=0.125kg | pins=4
    17. MT20

In [7]:
# ==============================================================================
# CELL 3: COORDINATE BOUNDS + NUMBA PENALTY-BREAKDOWN ENGINE
# ==============================================================================

# ---- Per-component (x_min,x_max,y_min,y_max) bounds so its footprint+clearance+pins
#      can never physically leave the active board area ----
lower_bounds = []
upper_bounds = []
for i in range(n):
    i_start = offset_indices[i]
    i_count = offset_counts[i]
    comp_offsets = flat_offsets[i_start : i_start + i_count]

    max_off_x = max(abs(x) for x, _ in comp_offsets) if i_count > 0 else 0.0
    max_off_y = max(abs(y) for _, y in comp_offsets) if i_count > 0 else 0.0

    c_top   = clearance_dirs[i, 0]
    c_right = clearance_dirs[i, 3] if is_back_side[i] else clearance_dirs[i, 1]
    c_bot   = clearance_dirs[i, 2]
    c_left  = clearance_dirs[i, 1] if is_back_side[i] else clearance_dirs[i, 3]

    req_margin_left   = max(lengths[i]/2.0 + c_left,  max_off_x + insert_diams[i]/2.0)
    req_margin_right  = max(lengths[i]/2.0 + c_right, max_off_x + insert_diams[i]/2.0)
    req_margin_bottom = max(widths[i]/2.0 + c_bot,    max_off_y + insert_diams[i]/2.0)
    req_margin_top    = max(widths[i]/2.0 + c_top,    max_off_y + insert_diams[i]/2.0)

    if not is_back_side[i]:
        x_min = -panel_width/2.0 + border_spacing + req_margin_left
        x_max =  panel_width/2.0 - border_spacing - req_margin_right
    else:
        x_min = -panel_width/2.0 + border_spacing + req_margin_right
        x_max =  panel_width/2.0 - border_spacing - req_margin_left

    y_min = -panel_height/2.0 + border_spacing + req_margin_bottom
    y_max =  panel_height/2.0 - border_spacing - req_margin_top

    lower_bounds.extend([x_min, y_min])
    upper_bounds.extend([x_max, y_max])

lower_bounds = np.array(lower_bounds, dtype=np.float64)
upper_bounds = np.array(upper_bounds, dtype=np.float64)

# Sanity check: bounds must not be inverted (component too big for board)
bad = np.where(lower_bounds > upper_bounds)[0]
if len(bad) > 0:
    print(f"[!] WARNING: {len(bad)} bound(s) inverted — component may not physically fit. Check board size / component sizes.")
else:
    print("[✓] All per-component coordinate bounds are valid.")

CG_TOLERANCE = 1.0  # mm

@njit(fastmath=True, cache=True)
def compute_penalty_breakdown(cx, cy, masses, hl, hw, req_d_matrix, num_components,
                               flat_offsets, offset_counts, offset_indices, is_back_side,
                               clearance_dirs, panel_width, panel_height, border_spacing):
    """
    Same geometric rules as the CMA-ES fitness core, but returns the 4 penalty
    components SEPARATELY (border, overlap, insert, cg) instead of one summed score,
    so the RL reward can weight priority-1 constraints vs priority-2 CG differently.
    """
    total_mass = 0.0
    cg_x = 0.0
    cg_y = 0.0
    abs_x = np.zeros(num_components)

    for i in range(num_components):
        m = masses[i]
        total_mass += m
        abs_x[i] = -cx[i] if is_back_side[i] else cx[i]
        cg_x += abs_x[i] * m
        cg_y += cy[i] * m

    if total_mass == 0.0:
        return 1e15, 1e15, 1e15, 1e15

    cg_x /= total_mass
    cg_y /= total_mass

    active_x_min = -panel_width/2.0 + border_spacing
    active_x_max =  panel_width/2.0 - border_spacing
    active_y_min = -panel_height/2.0 + border_spacing
    active_y_max =  panel_height/2.0 - border_spacing

    cg_offset = np.sqrt(cg_x**2 + cg_y**2)
    cg_penalty = 0.0
    if cg_offset > CG_TOLERANCE:
        cg_penalty = (cg_offset - CG_TOLERANCE) ** 2

    overlap_penalty = 0.0
    border_penalty = 0.0

    for i in range(num_components):
        c_top   = clearance_dirs[i, 0]
        c_right = clearance_dirs[i, 3] if is_back_side[i] else clearance_dirs[i, 1]
        c_bot   = clearance_dirs[i, 2]
        c_left  = clearance_dirs[i, 1] if is_back_side[i] else clearance_dirs[i, 3]

        i_x_min = abs_x[i] - hl[i] - c_left
        i_x_max = abs_x[i] + hl[i] + c_right
        i_y_min = cy[i] - hw[i] - c_bot
        i_y_max = cy[i] + hw[i] + c_top

        viol_left   = max(0.0, active_x_min - i_x_min)
        viol_right  = max(0.0, i_x_max - active_x_max)
        viol_bottom = max(0.0, active_y_min - i_y_min)
        viol_top    = max(0.0, i_y_max - active_y_max)

        total_border_viol = viol_left + viol_right + viol_bottom + viol_top
        if total_border_viol > 0.0:
            border_penalty += total_border_viol ** 2

        for j in range(i + 1, num_components):
            if is_back_side[i] == is_back_side[j]:
                cj_top   = clearance_dirs[j, 0]
                cj_right = clearance_dirs[j, 3] if is_back_side[j] else clearance_dirs[j, 1]
                cj_bot   = clearance_dirs[j, 2]
                cj_left  = clearance_dirs[j, 1] if is_back_side[j] else clearance_dirs[j, 3]

                j_x_min = abs_x[j] - hl[j] - cj_left
                j_x_max = abs_x[j] + hl[j] + cj_right
                j_y_min = cy[j] - hw[j] - cj_bot
                j_y_max = cy[j] + hw[j] + cj_top

                overlap_x = max(0.0, min(i_x_max, j_x_max) - max(i_x_min, j_x_min))
                overlap_y = max(0.0, min(i_y_max, j_y_max) - max(i_y_min, j_y_min))

                if overlap_x > 0.0 and overlap_y > 0.0:
                    overlap_area = overlap_x * overlap_y
                    overlap_penalty += overlap_area ** 2

    insert_penalty = 0.0
    for i in range(num_components):
        i_start = offset_indices[i]
        i_count = offset_counts[i]
        for j in range(i + 1, num_components):
            j_start = offset_indices[j]
            j_count = offset_counts[j]
            req_dist = req_d_matrix[i, j]

            for ii in range(i_count):
                idx_i = i_start + ii
                xi_abs = abs_x[i] + (-flat_offsets[idx_i, 0] if is_back_side[i] else flat_offsets[idx_i, 0])
                yi_abs = cy[i] + flat_offsets[idx_i, 1]

                for jj in range(j_count):
                    idx_j = j_start + jj
                    xj_abs = abs_x[j] + (-flat_offsets[idx_j, 0] if is_back_side[j] else flat_offsets[idx_j, 0])
                    yj_abs = cy[j] + flat_offsets[idx_j, 1]

                    dist = np.sqrt((xi_abs - xj_abs)**2 + (yi_abs - yj_abs)**2)
                    if req_dist > dist:
                        insert_penalty += (req_dist - dist) ** 2

    return border_penalty, overlap_penalty, insert_penalty, cg_penalty


def evaluate_layout(cx, cy):
    """Convenience wrapper: takes full coordinate arrays, returns penalty breakdown dict."""
    hl = lengths / 2.0
    hw = widths / 2.0
    b, o, ins, cg = compute_penalty_breakdown(
        cx, cy, masses, hl, hw, req_d_matrix, n,
        flat_offsets, offset_counts, offset_indices, is_back_side,
        clearance_dirs, panel_width, panel_height, border_spacing
    )
    return {"border": b, "overlap": o, "insert": ins, "cg": cg}


# Quick smoke test: evaluate at bounds midpoint (should still have some overlap most likely)
mid_cx = (lower_bounds[0::2] + upper_bounds[0::2]) / 2.0
mid_cy = (lower_bounds[1::2] + upper_bounds[1::2]) / 2.0
test_result = evaluate_layout(mid_cx, mid_cy)
print("[✓] Numba fitness engine compiled & tested. Sample penalty breakdown at bounds-midpoint layout:")
for k, v in test_result.items():
    print(f"    {k:8s}: {v:,.2f}")

[✓] All per-component coordinate bounds are valid.
[✓] Numba fitness engine compiled & tested. Sample penalty breakdown at bounds-midpoint layout:
    border  : 0.00
    overlap : 10,108,002,267.00
    insert  : 153,206.56
    cg      : 19.96


In [26]:
# ==============================================================================
# CELL 4: GYMNASIUM ENVIRONMENT (FINAL — annealed step, cx/cy in info to avoid
# SubprocVecEnv auto-reset race)
# ==============================================================================

MAX_STEP_MM   = 12.0     # nudge size at the START of an episode (coarse untangling)
MIN_STEP_MM   = 0.25     # nudge size by the END of an episode (fine insert-distance precision)
MAX_STEPS     = 250       # episode horizon (truncation)
STEP_COST     = 0.01      # tiny per-step penalty to encourage speed
SUCCESS_BONUS = 50.0

# Insert weighted highest (border/overlap already solvable easily); CG stays lowest (priority 2)
SCORE_WEIGHTS = {"border": 3.0, "overlap": 3.0, "insert": 10.0, "cg": 0.5}
VALID_EPS = 1e-6  # below this, a penalty term counts as "resolved"


def composite_score(breakdown):
    """Weighted sum of log1p-compressed penalty terms. Lower = better, 0 = perfect."""
    return (SCORE_WEIGHTS["border"]  * np.log1p(breakdown["border"]) +
            SCORE_WEIGHTS["overlap"] * np.log1p(breakdown["overlap"]) +
            SCORE_WEIGHTS["insert"]  * np.log1p(breakdown["insert"]) +
            SCORE_WEIGHTS["cg"]      * np.log1p(breakdown["cg"]))


def is_fully_valid(breakdown):
    return (breakdown["border"] < VALID_EPS and
            breakdown["overlap"] < VALID_EPS and
            breakdown["insert"] < VALID_EPS)


class PCBPlacementEnv(gym.Env):
    """
    Iterative-refinement placement environment.
    Episode: start from a random layout within per-component bounds.
    Each step: agent nudges every component's (x,y) simultaneously.
    Nudge size is ANNEALED within the episode: MAX_STEP_MM early (fast overlap
    resolution) -> MIN_STEP_MM late (fine insert-distance / CG tuning).
    Reward: reduction in composite penalty score (dense), + terminal bonus if fully valid.

    IMPORTANT: cx/cy are embedded directly into `info` at the end of step().
    Do NOT rely on a separate post-hoc state accessor (e.g. calling back into
    the env after the fact) to read cx/cy for "the layout that just finished" —
    SubprocVecEnv auto-resets a sub-env internally the instant an episode
    terminates/truncates, so by the time any external call reaches the worker
    process, self.cx/self.cy already belong to the NEXT episode. Reading cx/cy
    out of `info` inside this same step() call is the only race-free way.
    """
    metadata = {"render_modes": []}

    def __init__(self, seed=None):
        super().__init__()
        self.n = n
        self.lower = lower_bounds.astype(np.float32)
        self.upper = upper_bounds.astype(np.float32)
        self.range_ = (self.upper - self.lower)
        self.range_[self.range_ == 0] = 1.0

        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2 * self.n,), dtype=np.float32)
        obs_dim = 2 * self.n + 4 + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self.cx = None
        self.cy = None
        self.step_count = 0
        self._np_random_seed = seed

    def _normalize_pos(self):
        nx = 2.0 * (self.cx - self.lower[0::2]) / self.range_[0::2] - 1.0
        ny = 2.0 * (self.cy - self.lower[1::2]) / self.range_[1::2] - 1.0
        return np.stack([nx, ny], axis=1).ravel().astype(np.float32)

    def _get_breakdown(self):
        b, o, ins, cg = compute_penalty_breakdown(
            self.cx, self.cy, masses, lengths/2.0, widths/2.0, req_d_matrix, self.n,
            flat_offsets, offset_counts, offset_indices, is_back_side,
            clearance_dirs, panel_width, panel_height, border_spacing
        )
        return {"border": b, "overlap": o, "insert": ins, "cg": cg}

    def _get_obs(self, breakdown):
        pos = self._normalize_pos()
        pen = np.array([
            np.log1p(breakdown["border"]),
            np.log1p(breakdown["overlap"]),
            np.log1p(breakdown["insert"]),
            np.log1p(breakdown["cg"]),
        ], dtype=np.float32)
        step_frac = np.array([self.step_count / MAX_STEPS], dtype=np.float32)
        return np.concatenate([pos, pen, step_frac])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = self.np_random
        self.cx = rng.uniform(self.lower[0::2], self.upper[0::2]).astype(np.float32)
        self.cy = rng.uniform(self.lower[1::2], self.upper[1::2]).astype(np.float32)
        self.step_count = 0

        breakdown = self._get_breakdown()
        self.prev_score = composite_score(breakdown)
        obs = self._get_obs(breakdown)
        info = {"breakdown": breakdown}
        return obs, info

    def step(self, action):
        action = np.clip(action, -1.0, 1.0).astype(np.float32)

        progress = self.step_count / MAX_STEPS
        current_max_step = MAX_STEP_MM * (1.0 - progress) + MIN_STEP_MM * progress

        dx = action[0::2] * current_max_step
        dy = action[1::2] * current_max_step

        self.cx = np.clip(self.cx + dx, self.lower[0::2], self.upper[0::2])
        self.cy = np.clip(self.cy + dy, self.lower[1::2], self.upper[1::2])
        self.step_count += 1

        breakdown = self._get_breakdown()
        score = composite_score(breakdown)

        reward = (self.prev_score - score) - STEP_COST
        self.prev_score = score

        terminated = False
        if is_fully_valid(breakdown):
            reward += SUCCESS_BONUS
            terminated = True

        truncated = self.step_count >= MAX_STEPS
        obs = self._get_obs(breakdown)

        # cx/cy captured HERE, same process/same call, before any auto-reset can occur
        info = {"breakdown": breakdown, "score": score, "cx": self.cx.copy(), "cy": self.cy.copy()}

        return obs, float(reward), terminated, truncated, info


# ---- Sanity check ----
_test_env = PCBPlacementEnv(seed=SEED)
check_env(_test_env, warn=True)
print("[✓] PCBPlacementEnv passed gymnasium/SB3 env checker.")

obs, info = _test_env.reset(seed=SEED)
print(f"[✓] Reset OK. obs shape={obs.shape}, initial breakdown={ {k: round(v,2) for k,v in info['breakdown'].items()} }")
total_r = 0.0
for _ in range(5):
    a = _test_env.action_space.sample()
    obs, r, term, trunc, info = _test_env.step(a)
    total_r += r
print(f"[✓] 5 random steps ran fine. cumulative reward={total_r:.3f}, has cx/cy in info: {'cx' in info and 'cy' in info}")

[✓] PCBPlacementEnv passed gymnasium/SB3 env checker.
[✓] Reset OK. obs shape=(41,), initial breakdown={'border': 0.0, 'overlap': 1845082386.89, 'insert': 1331.69, 'cg': 12418.81}
[✓] 5 random steps ran fine. cumulative reward=-2.243, has cx/cy in info: True


In [27]:
# ==============================================================================
# CELL 5: VECTORIZED ENVIRONMENTS (envs only — model is built/loaded in Cell 6)
# ==============================================================================

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.monitor import Monitor

N_ENVS = 8

def make_env(rank, seed=SEED):
    def _init():
        env = PCBPlacementEnv(seed=seed + rank)
        env = Monitor(env)
        return env
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])

VECNORM_PATH = os.path.join(OUTPUT_DIR, "vecnormalize.pkl")
if os.path.exists(VECNORM_PATH):
    vec_env = VecNormalize.load(VECNORM_PATH, vec_env)
    print(f"[✓] Loaded existing VecNormalize stats from '{VECNORM_PATH}'.")
else:
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0, gamma=0.99)
    print("[✓] Created fresh VecNormalize (no existing stats found).")

vec_env.training = True
vec_env.norm_reward = True

print(f"[✓] {N_ENVS} parallel environments ready (SubprocVecEnv + VecNormalize).")

[✓] Loaded existing VecNormalize stats from 'optimized_layouts_rl/vecnormalize.pkl'.
[✓] 8 parallel environments ready (SubprocVecEnv + VecNormalize).


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [28]:
# ==============================================================================
# CELL 6: MODEL (LOAD-OR-CREATE) + FIXED CALLBACK + TRAINING
# ==============================================================================

MODEL_PATH = os.path.join(OUTPUT_DIR, "ppo_pcb_placement")

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
    activation_fn=torch.nn.Tanh,
)

if os.path.exists(MODEL_PATH + ".zip"):
    model = PPO.load(MODEL_PATH, env=vec_env, device=DEVICE)
    print(f"[✓] Loaded existing checkpoint from '{MODEL_PATH}.zip'.")
else:
    model = PPO(
        policy="MlpPolicy",
        env=vec_env,
        device=DEVICE,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        verbose=1,
        seed=SEED,
        tensorboard_log=os.path.join(OUTPUT_DIR, "tb_logs"),
    )
    print("[✓] Created fresh PPO model (no existing checkpoint found).")


class BestLayoutCallback(BaseCallback):
    """
    Tracks best fully-valid layout (lowest CG penalty) and best invalid layout
    (lowest composite score) across all parallel envs. cx/cy are read directly
    from info — populated inside env.step() BEFORE any SubprocVecEnv auto-reset
    can overwrite them. Do not fetch state via any other channel.
    """
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.best_valid_score = np.inf
        self.best_valid_layout = None
        self.best_invalid_score = np.inf
        self.best_invalid_layout = None
        self.n_valid_found = 0

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            breakdown = info.get("breakdown")
            if breakdown is None:
                continue
            score = composite_score(breakdown)
            if is_fully_valid(breakdown):
                self.n_valid_found += 1
                if breakdown["cg"] < self.best_valid_score:
                    self.best_valid_score = breakdown["cg"]
                    self.best_valid_layout = {"cx": info["cx"], "cy": info["cy"], "breakdown": dict(breakdown)}
            else:
                if score < self.best_invalid_score:
                    self.best_invalid_score = score
                    self.best_invalid_layout = {"cx": info["cx"], "cy": info["cy"], "breakdown": dict(breakdown)}

        if self.n_calls % 2000 == 0:
            self.logger.record("custom/best_valid_cg", self.best_valid_score if self.best_valid_layout else -1)
            self.logger.record("custom/best_invalid_score", self.best_invalid_score)
            self.logger.record("custom/n_valid_found", self.n_valid_found)
        return True


best_layout_cb = BestLayoutCallback(verbose=1)

TRAIN_TIMESTEPS = 1_000_000  # bump this and re-run the cell anytime to keep training the same checkpoint

print(f"[▶] Training for {TRAIN_TIMESTEPS:,} timesteps on device='{DEVICE}'...")
t0 = time.time()

model.learn(
    total_timesteps=TRAIN_TIMESTEPS,
    callback=best_layout_cb,
    progress_bar=True,
    reset_num_timesteps=False,
)

elapsed = time.time() - t0
print(f"\n[✓] Training complete in {elapsed/60:.1f} minutes.")
print(f"[✓] Fully valid layouts found this run: {best_layout_cb.n_valid_found}")
if best_layout_cb.best_valid_layout:
    print(f"[✓] Best valid layout CG penalty: {best_layout_cb.best_valid_score:.4f}")
    print(f"    Full breakdown: {best_layout_cb.best_valid_layout['breakdown']}")
else:
    print(f"[!] No fully valid layout this run. Best invalid composite score: {best_layout_cb.best_invalid_score:.4f}")
    print(f"    Breakdown: {best_layout_cb.best_invalid_layout['breakdown']}")

model.save(MODEL_PATH)
vec_env.save(VECNORM_PATH)
print(f"[✓] Model and VecNormalize stats saved to '{OUTPUT_DIR}/'.")

[✓] Loaded existing checkpoint from 'optimized_layouts_rl/ppo_pcb_placement.zip'.
[▶] Training for 1,000,000 timesteps on device='mps'...
Logging to optimized_layouts_rl/tb_logs/PPO_3
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 250      |
|    ep_rew_mean     | 20.3     |
| time/              |          |
|    fps             | 2853     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2531328  |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    best_invalid_score   | 58.6        |
|    best_valid_cg        | -1          |
|    n_valid_found        | 0           |
| rollout/                |             |
|    ep_len_mean          | 250         |
|    ep_rew_mean          | 10.4        |
| time/                   |             |
|    fps                  | 2249        |
|    iterations           | 2           |
|    time_elap


[✓] Training complete in 8.7 minutes.
[✓] Fully valid layouts found this run: 20
[✓] Best valid layout CG penalty: 16.4566
    Full breakdown: {'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 16.456634779965544}
[✓] Model and VecNormalize stats saved to 'optimized_layouts_rl/'.


In [29]:
# ==============================================================================
# CELL 7: EXTRACT BEST LAYOUT, VERIFY, AND RENDER (MATCHING ORIGINAL CMA-ES STYLE)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import pandas as pd

# ---- 1. Extract best valid layout found during training ----
assert best_layout_cb.best_valid_layout is not None, "No valid layout found — cannot proceed to Cell 7 yet."

best_cx = best_layout_cb.best_valid_layout["cx"].copy()
best_cy = best_layout_cb.best_valid_layout["cy"].copy()

# ---- 2. Independent re-verification (never trust training-time numbers blindly) ----
verify = evaluate_layout(best_cx, best_cy)
print("[✓] Independent re-verification of best layout:")
for k, v in verify.items():
    print(f"    {k:8s}: {v:,.4f}")
assert is_fully_valid(verify), "Re-verification FAILED — layout is not actually valid. Investigate before using."
print("[✓] Layout confirmed fully valid (border/overlap/insert all ~0).")

# ---- 3. Compute derived quantities for rendering ----
def get_absolute_pin_positions(cx, cy):
    """Returns list of (component_idx, pin_x_abs, pin_y_abs) for every insert pin."""
    pins = []
    for i in range(n):
        i_start = offset_indices[i]
        i_count = offset_counts[i]
        for k in range(i_count):
            ox, oy = flat_offsets[i_start + k]
            px = cx[i] + (-ox if is_back_side[i] else ox)
            py = cy[i] + oy
            pins.append((i, px, py))
    return pins

def get_close_pin_pairs(cx, cy, threshold=40.0):
    """Find pin pairs across DIFFERENT components under `threshold` mm apart."""
    pins = get_absolute_pin_positions(cx, cy)
    close_pairs = []
    for a in range(len(pins)):
        ia, xa, ya = pins[a]
        for b in range(a + 1, len(pins)):
            ib, xb, yb = pins[b]
            if ia == ib:
                continue
            d = np.sqrt((xa - xb)**2 + (ya - yb)**2)
            if d < threshold:
                close_pairs.append((ia, ib, xa, ya, xb, yb, d))
    return close_pairs

def abs_x_of(cx, i):
    return -cx[i] if is_back_side[i] else cx[i]

# CG (mirrored X for back components, matching original convention)
total_mass = masses.sum()
cg_x = sum(abs_x_of(best_cx, i) * masses[i] for i in range(n)) / total_mass
cg_y = sum(best_cy[i] * masses[i] for i in range(n)) / total_mass
print(f"[✓] Combined Assembly CG offset: ({cg_x:.4f}, {cg_y:.4f}) mm")

# ---- 4. Plot: FRONT/BACK combined view (style of layout_5_main.png) ----
def draw_layout_panel(ax, side_is_back, title):
    active_x_min = -panel_width/2 + border_spacing
    active_x_max =  panel_width/2 - border_spacing
    active_y_min = -panel_height/2 + border_spacing
    active_y_max =  panel_height/2 - border_spacing

    ax.add_patch(mpatches.Rectangle((-panel_width/2, -panel_height/2), panel_width, panel_height,
                                     facecolor='none', edgecolor='none'))
    # Border strip
    border_outer = mpatches.Rectangle((-panel_width/2, -panel_height/2), panel_width, panel_height,
                                       facecolor='#f8d7da', edgecolor='none', alpha=0.6, zorder=0)
    border_inner = mpatches.Rectangle((active_x_min, active_y_min),
                                       active_x_max - active_x_min, active_y_max - active_y_min,
                                       facecolor='white', edgecolor='none', zorder=1)
    ax.add_patch(border_outer)
    ax.add_patch(border_inner)

    for i in range(n):
        is_this_side = (is_back_side[i] == 1) if side_is_back else (is_back_side[i] == 0)
        if not is_this_side:
            continue

        x_abs = abs_x_of(best_cx, i)
        y = best_cy[i]
        hl, hw = lengths[i]/2, widths[i]/2

        c_top   = clearance_dirs[i, 0]
        c_right = clearance_dirs[i, 3] if is_back_side[i] else clearance_dirs[i, 1]
        c_bot   = clearance_dirs[i, 2]
        c_left  = clearance_dirs[i, 1] if is_back_side[i] else clearance_dirs[i, 3]

        # Clearance zone (dotted light green)
        ax.add_patch(mpatches.Rectangle(
            (x_abs - hl - c_left, y - hw - c_bot), hl*2 + c_left + c_right, hw*2 + c_top + c_bot,
            facecolor='#d4edda', edgecolor='#6c9', linestyle=':', linewidth=0.8, alpha=0.5, zorder=2))

        color = '#2e6ea6' if not side_is_back else '#1a5c1a'
        if element_shapes[i] == 'rectangle':
            ax.add_patch(mpatches.Rectangle((x_abs - hl, y - hw), hl*2, hw*2,
                                             facecolor=color, edgecolor='black', linewidth=0.6,
                                             alpha=0.85, zorder=3))
        else:
            ax.add_patch(mpatches.Circle((x_abs, y), hl, facecolor=color, edgecolor='black',
                                          linewidth=0.6, alpha=0.85, zorder=3))
            ax.add_patch(mpatches.Circle((x_abs, y), hl + insert_diams[i], facecolor='#cfe2f3',
                                          edgecolor='#e06090', linestyle='--', linewidth=0.7,
                                          alpha=0.4, zorder=2))

        ax.text(x_abs, y, element_names[i], ha='center', va='center',
                fontsize=7, fontweight='bold', color='white', zorder=4)

        # Insert pins
        i_start = offset_indices[i]
        for k in range(offset_counts[i]):
            ox, oy = flat_offsets[i_start + k]
            px = x_abs + (-ox if is_back_side[i] else ox)
            py = y + oy
            ax.plot(px, py, 'o', color='red', markersize=3, zorder=5)

    ax.set_xlim(-panel_width/2 - 20, panel_width/2 + 20)
    ax.set_ylim(-panel_height/2 - 20, panel_height/2 + 20)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.4)


fig1, axes = plt.subplots(1, 2, figsize=(18, 9))
fig1.suptitle(f"PPO-Optimized Layout (Best Valid, {best_layout_cb.n_valid_found} found)\n"
              f"Combined Assembly CG Offset: ({cg_x:.4f}, {cg_y:.4f}) mm", fontsize=15, fontweight='bold')
draw_layout_panel(axes[0], side_is_back=False, title="FRONT SIDE VIEW")
draw_layout_panel(axes[1], side_is_back=True, title="BACK SIDE VIEW (FLIPPED)")

legend_elems = [mpatches.Patch(facecolor='#2e6ea6', label='Front Layer Components'),
                mpatches.Patch(facecolor='#1a5c1a', label='Back Layer Components')]
fig1.legend(handles=legend_elems, loc='lower center', ncol=2, fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig1_path = os.path.join(OUTPUT_DIR, "ppo_layout_front_back.png")
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[✓] Saved: {fig1_path}")

# ---- 5. Plot: Proximity / pin-distance verification (style of layout_5_distance_map.png) ----
close_pairs = get_close_pin_pairs(best_cx, best_cy, threshold=40.0)

fig2, ax2 = plt.subplots(figsize=(16, 11))
fig2.suptitle("PPO-Optimized Layout — Inter-Insert Proximity Verification (< 40mm)\n(Front Side Projection View)",
              fontsize=14, fontweight='bold')

active_x_min = -panel_width/2 + border_spacing
active_x_max =  panel_width/2 - border_spacing
active_y_min = -panel_height/2 + border_spacing
active_y_max =  panel_height/2 - border_spacing
ax2.add_patch(mpatches.Rectangle((-panel_width/2, -panel_height/2), panel_width, panel_height,
                                  facecolor='#f8d7da', alpha=0.5, zorder=0))
ax2.add_patch(mpatches.Rectangle((active_x_min, active_y_min), active_x_max-active_x_min, active_y_max-active_y_min,
                                  facecolor='#f5f5f5', zorder=1))

for i in range(n):
    x_abs = abs_x_of(best_cx, i)
    y = best_cy[i]
    hl, hw = lengths[i]/2, widths[i]/2
    is_back = is_back_side[i] == 1
    face_color = '#a8c8e8' if not is_back else '#c8e6c9'
    edge_style = '--' if is_back else '-'

    if element_shapes[i] == 'rectangle':
        ax2.add_patch(mpatches.Rectangle((x_abs - hl, y - hw), hl*2, hw*2, facecolor=face_color,
                                          edgecolor='green' if is_back else 'steelblue',
                                          linestyle=edge_style, linewidth=1.2, alpha=0.7, zorder=2))
    else:
        ax2.add_patch(mpatches.Circle((x_abs, y), hl, facecolor=face_color,
                                       edgecolor='green' if is_back else 'steelblue',
                                       linestyle=edge_style, linewidth=1.2, alpha=0.7, zorder=2))
    ax2.text(x_abs, y, element_names[i], ha='center', va='center', fontsize=7, fontweight='bold', zorder=4)

    i_start = offset_indices[i]
    for k in range(offset_counts[i]):
        ox, oy = flat_offsets[i_start + k]
        px = x_abs + (-ox if is_back_side[i] else ox)
        py = y + oy
        ax2.plot(px, py, 'o', color='red', markersize=3, zorder=5)

# Draw close-pair connector lines with labels
summary_lines = []
label_letters = 'abcdefghijklmnopqrstuvwxyz'
for idx, (ia, ib, xa, ya, xb, yb, d) in enumerate(close_pairs):
    letter = label_letters[idx % len(label_letters)]
    ax2.plot([xa, xb], [ya, yb], linestyle='--', color='darkgreen', linewidth=1.2, zorder=6)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ax2.plot(mx, my, 'o', color='purple', markersize=14, zorder=7)
    ax2.text(mx, my, letter, ha='center', va='center', fontsize=8, color='white',
             fontweight='bold', zorder=8)
    summary_lines.append(f"{letter} — {d:.2f} mm ({element_names[ia]} ↔ {element_names[ib]})")

ax2.set_xlim(-panel_width/2 - 20, panel_width/2 + 20)
ax2.set_ylim(-panel_height/2 - 20, panel_height/2 + 20)
ax2.set_aspect('equal')
ax2.grid(True, linestyle=':', alpha=0.4)

summary_text = "Pin Distances Summary:\n\n" + "\n".join(summary_lines) if summary_lines else "Pin Distances Summary:\n\nNone under 40mm"
ax2.text(1.02, 0.75, summary_text, transform=ax2.transAxes, fontsize=9,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='#f0f0f0', edgecolor='gray'))

plt.tight_layout()
fig2_path = os.path.join(OUTPUT_DIR, "ppo_layout_proximity_map.png")
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[✓] Saved: {fig2_path}")

# ---- 6. Data table (component-level summary) ----
table_rows = []
for i in range(n):
    x_abs = abs_x_of(best_cx, i)
    y = best_cy[i]
    dims = f"{lengths[i]:.1f} x {widths[i]:.1f}" if element_shapes[i] == 'rectangle' else f"Ø {lengths[i]:.1f}"
    table_rows.append({
        "Component Name": element_names[i],
        "Layer Placement": "BACK" if is_back_side[i] else "FRONT",
        "Mass (kg)": round(masses[i], 3),
        "Dimensions (mm)": dims,
        "CoG X (mm)": round(x_abs, 2),
        "CoG Y (mm)": round(y, 2),
    })

df = pd.DataFrame(table_rows)
csv_path = os.path.join(OUTPUT_DIR, "ppo_layout_table.csv")
df.to_csv(csv_path, index=False)
print(f"[✓] Saved: {csv_path}")
print(df.to_string(index=False))

[✓] Independent re-verification of best layout:
    border  : 0.0000
    overlap : 0.0000
    insert  : 0.0000
    cg      : 16.4566
[✓] Layout confirmed fully valid (border/overlap/insert all ~0).
[✓] Combined Assembly CG offset: (-3.9408, -3.1686) mm
[✓] Saved: optimized_layouts_rl/ppo_layout_front_back.png
[✓] Saved: optimized_layouts_rl/ppo_layout_proximity_map.png
[✓] Saved: optimized_layouts_rl/ppo_layout_table.csv
Component Name Layer Placement  Mass (kg) Dimensions (mm)  CoG X (mm)  CoG Y (mm)
           PDS           FRONT      1.500   100.0 x 112.0 -147.289993   44.560001
         CCRR0           FRONT      0.150          Ø 45.0 -297.130005  208.750000
         CCRR1           FRONT      0.150          Ø 45.0  311.839996   18.620001
           LD0           FRONT      0.040          Ø 32.0  424.000000 -162.630005
           LD1           FRONT      0.040          Ø 32.0 -284.589996 -235.199997
           LD2           FRONT      0.040          Ø 32.0 -287.660004   58.529999
 

In [31]:
# ==============================================================================
# CELL 8: MULTI-LAYOUT GENERATION — collect up to 5 distinct valid layouts by
# rolling out the TRAINED policy from independent random resets (inference
# only — model.learn() is never called here). Equivalent to what the old
# CMA-ES script did with multiple restarts.
#
# NOTE: training's success rate was ~0.14% per episode (20 valid / ~14,000
# episodes), so this needs many more attempts than a first guess suggests —
# run in parallel (8 envs, like training) and use the deterministic policy,
# which is typically far more reliable at inference than the sampling policy
# used during training exploration.
# ==============================================================================

from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize as VecNormalizeInfer

N_PARALLEL      = 8
N_EPISODES      = 800     # target number of completed episodes across all envs
MAX_LAYOUTS     = 5        # cap, matching original CMA-ES output count
DEDUP_DIST_MM   = 25.0     # mean per-component (x,y) shift below this = "same" layout
DETERMINISTIC   = True     # mean action, no exploration noise — more reliable at inference
VECNORM_PATH    = os.path.join(OUTPUT_DIR, "vecnormalize.pkl")

assert 'model' in dir(), "No trained model in memory — run Cell 6 first (or load checkpoint)."
assert os.path.exists(VECNORM_PATH), f"Missing {VECNORM_PATH} — run Cell 6 first."

def make_single_env():
    return PCBPlacementEnv()

rollout_venv = DummyVecEnv([make_single_env for _ in range(N_PARALLEL)])
rollout_venv = VecNormalizeInfer.load(VECNORM_PATH, rollout_venv)
rollout_venv.training = False     # freeze running obs/reward stats — inference only
rollout_venv.norm_reward = False

rollout_venv.seed(SEED + 1000)
obs = rollout_venv.reset()

candidates = []       # list of dicts: cx, cy, breakdown, cg
episodes_done = 0
t0 = time.time()

while episodes_done < N_EPISODES:
    action, _ = model.predict(obs, deterministic=DETERMINISTIC)
    obs, reward, done, infos = rollout_venv.step(action)
    for i, info in enumerate(infos):
        if done[i]:
            episodes_done += 1
            cx, cy, breakdown = info.get("cx"), info.get("cy"), info.get("breakdown")
            if cx is not None and is_fully_valid(breakdown):
                verify = evaluate_layout(cx, cy)   # independent re-verification
                if is_fully_valid(verify):
                    candidates.append({"cx": cx, "cy": cy, "breakdown": verify, "cg": verify["cg"]})

print(f"[✓] {episodes_done} episodes completed in {time.time()-t0:.1f}s.")
print(f"[✓] {len(candidates)} / {episodes_done} rollouts produced an independently-verified valid layout "
      f"({100*len(candidates)/max(episodes_done,1):.3f}% success rate).")

if len(candidates) == 0:
    print("[!] Still 0 valid layouts. Options: raise N_EPISODES further, or extend training "
          "(re-run Cell 6 with more TRAIN_TIMESTEPS) — the underlying success rate may just be "
          "too low right now for a practical rollout count.")
else:
    # ---- Sort by CG penalty (best first), then greedily deduplicate ----
    candidates.sort(key=lambda c: c["cg"])

    def mean_shift_mm(a, b):
        dx = a["cx"] - b["cx"]
        dy = a["cy"] - b["cy"]
        return np.sqrt(dx**2 + dy**2).mean()

    distinct_layouts = []
    for cand in candidates:
        if all(mean_shift_mm(cand, kept) >= DEDUP_DIST_MM for kept in distinct_layouts):
            distinct_layouts.append(cand)
        if len(distinct_layouts) >= MAX_LAYOUTS:
            break

    print(f"[✓] {len(distinct_layouts)} distinct valid layout(s) kept (deduped at {DEDUP_DIST_MM}mm mean shift):")
    for idx, lay in enumerate(distinct_layouts):
        print(f"    Layout {idx+1}: CG penalty={lay['cg']:.4f}, "
              f"breakdown={ {k: round(v,4) for k,v in lay['breakdown'].items()} }")

[✓] 800 episodes completed in 26.1s.
[✓] 22 / 800 rollouts produced an independently-verified valid layout (2.750% success rate).
[✓] 5 distinct valid layout(s) kept (deduped at 25.0mm mean shift):
    Layout 1: CG penalty=12.2026, breakdown={'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 12.2026}
    Layout 2: CG penalty=18.6801, breakdown={'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 18.6801}
    Layout 3: CG penalty=44.3006, breakdown={'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 44.3006}
    Layout 4: CG penalty=75.8543, breakdown={'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 75.8543}
    Layout 5: CG penalty=80.6805, breakdown={'border': 0.0, 'overlap': 0.0, 'insert': 0.0, 'cg': 80.6805}


In [39]:
# ==============================================================================
# CELL 9: RENDER ALL DISTINCT LAYOUTS – CMA‑ES STYLE (Main + Distance Map)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import os

assert 'distinct_layouts' in dir() and len(distinct_layouts) > 0, "Run Cell 8 first."

MULTI_DIR = os.path.join(OUTPUT_DIR, "multi_layouts")
os.makedirs(MULTI_DIR, exist_ok=True)

def abs_x_of_layout(cx, i):
    return -cx[i] if is_back_side[i] else cx[i]

def render_layout(layout_idx, cx, cy, cg_x, cg_y):
    """
    Render two figures per layout:
    1) Front/Back side view (main)
    2) Distance map with table (proximity)
    """
    num_elements = n
    total_mass = masses.sum()
    
    # -------------------- FIGURE 1: FRONT & BACK SIDE VIEWS --------------------
    fig, (ax_f, ax_b) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f"Layout Alternative {layout_idx} (PPO)\nCombined Assembly CG Offset: ({cg_x:.4f}, {cg_y:.4f}) mm",
                 fontsize=14, fontweight='bold')

    # Board outline and red border margin strips
    for ax in [ax_f, ax_b]:
        ax.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2), panel_width, panel_height,
                                   color='lightgray', alpha=0.3))
        # Red margin strips (top, bottom, left, right)
        ax.add_patch(plt.Rectangle((-panel_width/2, panel_height/2 - border_spacing),
                                   panel_width, border_spacing, color='red', alpha=0.15))
        ax.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2),
                                   panel_width, border_spacing, color='red', alpha=0.15))
        ax.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2 + border_spacing),
                                   border_spacing, panel_height - 2*border_spacing, color='red', alpha=0.15))
        ax.add_patch(plt.Rectangle((panel_width/2 - border_spacing, -panel_height/2 + border_spacing),
                                   border_spacing, panel_height - 2*border_spacing, color='red', alpha=0.15))

    # Draw components on Front and Back panels
    for i in range(num_elements):
        ins_rad = insert_diams[i] / 2.0
        i_start = offset_indices[i]
        c_top, c_right, c_bot, c_left = clearance_dirs[i]

        # ---- FRONT components ----
        if not is_back_side[i]:
            ex, ey = cx[i], cy[i]
            # Clearance envelope
            if element_shapes[i] == 'rectangle':
                c_x = ex - lengths[i]/2.0 - c_left
                c_y = ey - widths[i]/2.0 - c_bot
                c_w = lengths[i] + c_left + c_right
                c_h = widths[i] + c_top + c_bot
                ax_f.add_patch(plt.Rectangle((c_x, c_y), c_w, c_h,
                                             facecolor='#A9C7EB', edgecolor='crimson',
                                             linestyle='--', linewidth=1.2, alpha=0.5))
                ax_f.add_patch(plt.Rectangle((ex - lengths[i]/2, ey - widths[i]/2),
                                             lengths[i], widths[i],
                                             color='royalblue', alpha=0.9,
                                             edgecolor='navy', linewidth=1.5))
            else:
                ax_f.add_patch(plt.Circle((ex, ey), lengths[i]/2 + clearance,
                                          facecolor='#A9C7EB', edgecolor='crimson',
                                          linestyle='--', linewidth=1.2, alpha=0.5))
                ax_f.add_patch(plt.Circle((ex, ey), lengths[i]/2,
                                          color='royalblue', alpha=0.9,
                                          edgecolor='navy', linewidth=1.5))

            # Pins (front – red)
            for ii in range(offset_counts[i]):
                dx, dy = flat_offsets[i_start + ii]
                ax_f.add_patch(plt.Circle((ex + dx, ey + dy), ins_rad,
                                          color='crimson', zorder=4))
                # Back panel shows mirrored pin positions (dashed)
                ax_b.add_patch(plt.Circle((-(ex + dx), ey + dy), ins_rad,
                                          facecolor='crimson', edgecolor='#5C0612',
                                          linewidth=1.5, alpha=0.6, zorder=3))

            # Label
            ax_f.text(ex, ey, element_names[i], color='white', ha='center', va='center',
                      fontsize=8, fontweight='bold', zorder=5)

            # Trace outline on back panel
            ex_trace_b = -cx[i]
            if element_shapes[i] == 'rectangle':
                ax_b.add_patch(plt.Rectangle((ex_trace_b - lengths[i]/2, cy[i] - widths[i]/2),
                                             lengths[i], widths[i], fill=False,
                                             linestyle='--', edgecolor='navy', linewidth=1.2, alpha=0.7))
            else:
                ax_b.add_patch(plt.Circle((ex_trace_b, cy[i]), lengths[i]/2,
                                          fill=False, linestyle='--',
                                          edgecolor='navy', linewidth=1.2, alpha=0.7))

        # ---- BACK components ----
        else:
            ex_b, ey_b = cx[i], cy[i]
            # Clearance envelope on back panel
            if element_shapes[i] == 'rectangle':
                c_x = ex_b - lengths[i]/2.0 - c_left
                c_y = ey_b - widths[i]/2.0 - c_bot
                c_w = lengths[i] + c_left + c_right
                c_h = widths[i] + c_top + c_bot
                ax_b.add_patch(plt.Rectangle((c_x, c_y), c_w, c_h,
                                             facecolor='#A3D1A3', edgecolor='darkgreen',
                                             linestyle='--', linewidth=1.2, alpha=0.5))
                ax_b.add_patch(plt.Rectangle((ex_b - lengths[i]/2, ey_b - widths[i]/2),
                                             lengths[i], widths[i],
                                             color='darkgreen', alpha=0.9,
                                             edgecolor='darkslategrey', linewidth=1.5))
            else:
                ax_b.add_patch(plt.Circle((ex_b, ey_b), lengths[i]/2 + clearance,
                                          facecolor='#A3D1A3', edgecolor='darkgreen',
                                          linestyle='--', linewidth=1.2, alpha=0.5))
                ax_b.add_patch(plt.Circle((ex_b, ey_b), lengths[i]/2,
                                          color='darkgreen', alpha=0.9,
                                          edgecolor='darkslategrey', linewidth=1.5))

            # Pins (back – orange)
            for ii in range(offset_counts[i]):
                dx, dy = flat_offsets[i_start + ii]
                ax_b.add_patch(plt.Circle((ex_b + dx, ey_b + dy), ins_rad,
                                          color='orange', zorder=4))
                ax_f.add_patch(plt.Circle((-(ex_b + dx), ey_b + dy), ins_rad,
                                          facecolor='orange', edgecolor='#733D00',
                                          linewidth=1.5, alpha=0.6, zorder=3))

            # Label
            ax_b.text(ex_b, ey_b, element_names[i], color='white', ha='center', va='center',
                      fontsize=8, fontweight='bold', zorder=5)

            # Trace outline on front panel
            ex_trace_f = -cx[i]
            if element_shapes[i] == 'rectangle':
                ax_f.add_patch(plt.Rectangle((ex_trace_f - lengths[i]/2, cy[i] - widths[i]/2),
                                             lengths[i], widths[i], fill=False,
                                             linestyle='--', edgecolor='darkslategrey',
                                             linewidth=1.2, alpha=0.7))
            else:
                ax_f.add_patch(plt.Circle((ex_trace_f, cy[i]), lengths[i]/2,
                                          fill=False, linestyle='--',
                                          edgecolor='darkslategrey', linewidth=1.2, alpha=0.7))

    # Format axes
    for name, ax in [("FRONT SIDE VIEW", ax_f), ("BACK SIDE VIEW (FLIPPED)", ax_b)]:
        ax.set_title(name, fontweight='bold', fontsize=11)
        ax.set_xlim(-panel_width/2 - 10, panel_width/2 + 10)
        ax.set_ylim(-panel_height/2 - 10, panel_height/2 + 10)
        ax.set_aspect('equal')
        ax.grid(True, linestyle=':', alpha=0.5)

    plt.tight_layout()
    main_path = os.path.join(MULTI_DIR, f"layout_{layout_idx}_main.png")
    plt.savefig(main_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

    # -------------------- FIGURE 2: DISTANCE MAP WITH TABLE --------------------
    fig_c = plt.figure(figsize=(16, 11))
    gs_c = fig_c.add_gridspec(2, 1, height_ratios=[6.5, 3.5], hspace=0.25)

    ax_c = fig_c.add_subplot(gs_c[0, 0])
    ax_table = fig_c.add_subplot(gs_c[1, 0])
    ax_table.axis('off')

    fig_c.suptitle(f"Layout Alternative {layout_idx} (PPO) — Inter-Insert Proximity Verification (< 40mm)\n(Front Side Projection View)",
                   fontsize=13, fontweight='bold')

    # Board background with red margin strips
    ax_c.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2), panel_width, panel_height,
                                 color='lightgray', alpha=0.2))
    ax_c.add_patch(plt.Rectangle((-panel_width/2, panel_height/2 - border_spacing),
                                 panel_width, border_spacing, color='red', alpha=0.08))
    ax_c.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2),
                                 panel_width, border_spacing, color='red', alpha=0.08))
    ax_c.add_patch(plt.Rectangle((-panel_width/2, -panel_height/2 + border_spacing),
                                 border_spacing, panel_height - 2*border_spacing, color='red', alpha=0.08))
    ax_c.add_patch(plt.Rectangle((panel_width/2 - border_spacing, -panel_height/2 + border_spacing),
                                 border_spacing, panel_height - 2*border_spacing, color='red', alpha=0.08))

    # Collect data for table
    table_data = []
    abs_x_positions = np.where(is_back_side == 1, -cx, cx)

    for i in range(num_elements):
        i_start = offset_indices[i]
        ins_rad = insert_diams[i] / 2.0
        ey = cy[i]
        c_top, c_right, c_bot, c_left = clearance_dirs[i]
        side_str = "BACK" if is_back_side[i] else "FRONT"
        front_view_x = abs_x_positions[i]
        front_view_y = cy[i]

        table_data.append([
            element_names[i],
            side_str,
            f"{masses[i]:.3f}",
            f"{lengths[i]:.1f} x {widths[i]:.1f}" if element_shapes[i] == 'rectangle' else f"Ø {lengths[i]:.1f}",
            f"{front_view_x:.2f}",
            f"{front_view_y:.2f}"
        ])

        # Draw components on proximity map
        if not is_back_side[i]:
            ex = cx[i]
            # Clearance envelope (light blue, crimson dashed)
            if element_shapes[i] == 'rectangle':
                c_x = ex - lengths[i]/2.0 - c_left
                c_y = ey - widths[i]/2.0 - c_bot
                c_w = lengths[i] + c_left + c_right
                c_h = widths[i] + c_top + c_bot
                ax_c.add_patch(plt.Rectangle((c_x, c_y), c_w, c_h,
                                             facecolor='#A9C7EB', edgecolor='crimson',
                                             linestyle='--', linewidth=1.2, alpha=0.35))
                ax_c.add_patch(plt.Rectangle((ex - lengths[i]/2, ey - widths[i]/2),
                                             lengths[i], widths[i],
                                             color='royalblue', alpha=0.75,
                                             edgecolor='navy', linewidth=1.5))
            else:
                ax_c.add_patch(plt.Circle((ex, ey), lengths[i]/2 + clearance,
                                          facecolor='#A9C7EB', edgecolor='crimson',
                                          linestyle='--', linewidth=1.2, alpha=0.35))
                ax_c.add_patch(plt.Circle((ex, ey), lengths[i]/2,
                                          color='royalblue', alpha=0.75,
                                          edgecolor='navy', linewidth=1.5))

            # Pins (red)
            for ii in range(offset_counts[i]):
                dx, dy = flat_offsets[i_start + ii]
                ax_c.add_patch(plt.Circle((ex + dx, ey + dy), ins_rad,
                                          color='crimson', zorder=4))
            ax_c.text(ex, ey, element_names[i], color='white', ha='center', va='center',
                      fontsize=8, fontweight='bold', zorder=5)

        else:  # BACK components
            ex = -cx[i]   # mirrored X for front projection
            # Clearance envelope (light green, dotted)
            # Note: clearance directions are swapped for back when projected
            c_left_proj, c_right_proj = c_right, c_left
            if element_shapes[i] == 'rectangle':
                c_x = ex - lengths[i]/2.0 - c_left_proj
                c_y = ey - widths[i]/2.0 - c_bot
                c_w = lengths[i] + c_left_proj + c_right_proj
                c_h = widths[i] + c_top + c_bot
                ax_c.add_patch(plt.Rectangle((c_x, c_y), c_w, c_h,
                                             facecolor='#A3D1A3', edgecolor='darkgreen',
                                             linestyle=':', linewidth=1.2, alpha=0.35))
                ax_c.add_patch(plt.Rectangle((ex - lengths[i]/2, ey - widths[i]/2),
                                             lengths[i], widths[i],
                                             fill=False, linestyle='--',
                                             edgecolor='darkgreen', linewidth=1.5, zorder=3))
            else:
                ax_c.add_patch(plt.Circle((ex, ey), lengths[i]/2 + clearance,
                                          facecolor='#A3D1A3', edgecolor='darkgreen',
                                          linestyle=':', linewidth=1.2, alpha=0.35))
                ax_c.add_patch(plt.Circle((ex, ey), lengths[i]/2,
                                          fill=False, linestyle='--',
                                          edgecolor='darkgreen', linewidth=1.5, zorder=3))

            # Pins (orange)
            for ii in range(offset_counts[i]):
                dx, dy = flat_offsets[i_start + ii]
                # Mirror dx as well
                ax_c.add_patch(plt.Circle((ex - dx, ey + dy), ins_rad,
                                          facecolor='orange', edgecolor='#733D00',
                                          linewidth=1.2, alpha=0.85, zorder=4))
            ax_c.text(ex, ey, element_names[i], color='darkgreen', ha='center', va='center',
                      fontsize=8, fontweight='bold', zorder=5)

    # Find and draw close pin pairs (< 40mm)
    close_pairs_labels = []
    label_counter = 0
    alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"

    for i in range(num_elements):
        i_start = offset_indices[i]
        for j in range(i + 1, num_elements):
            j_start = offset_indices[j]
            for ii in range(offset_counts[i]):
                idx_i = i_start + ii
                xi_abs = abs_x_positions[i] + (-flat_offsets[idx_i, 0] if is_back_side[i] else flat_offsets[idx_i, 0])
                yi_abs = cy[i] + flat_offsets[idx_i, 1]
                for jj in range(offset_counts[j]):
                    idx_j = j_start + jj
                    xj_abs = abs_x_positions[j] + (-flat_offsets[idx_j, 0] if is_back_side[j] else flat_offsets[idx_j, 0])
                    yj_abs = cy[j] + flat_offsets[idx_j, 1]
                    dist = np.sqrt((xi_abs - xj_abs)**2 + (yi_abs - yj_abs)**2)
                    if dist < 40.0:
                        # Compute projected coordinates for line
                        xf1 = cx[i] + flat_offsets[idx_i, 0] if not is_back_side[i] else -(cx[i] + flat_offsets[idx_i, 0])
                        yf1 = cy[i] + flat_offsets[idx_i, 1]
                        xf2 = cx[j] + flat_offsets[idx_j, 0] if not is_back_side[j] else -(cx[j] + flat_offsets[idx_j, 0])
                        yf2 = cy[j] + flat_offsets[idx_j, 1]
                        letter = alphabet[label_counter % len(alphabet)]
                        label_counter += 1
                        ax_c.plot([xf1, xf2], [yf1, yf2], color='purple', linestyle='--',
                                  linewidth=1.5, alpha=0.85, zorder=10)
                        ax_c.text((xf1 + xf2)/2, (yf1 + yf2)/2, letter,
                                  color='white', fontsize=8, fontweight='bold',
                                  ha='center', va='center',
                                  bbox=dict(boxstyle="circle,pad=0.2", fc="darkmagenta",
                                            ec="none", alpha=0.9), zorder=11)
                        close_pairs_labels.append(f"{letter} — {dist:.2f} mm ({element_names[i]} ↔ {element_names[j]})")

    # Legend and summary
    if close_pairs_labels:
        legend_text = "Pin Distances Summary:\n\n" + "\n".join(close_pairs_labels)
    else:
        legend_text = "Pin Distances Summary:\n\nNo insert violations\nor pins found under 40mm."
    ax_c.text(1.02, 0.95, legend_text,
              transform=ax_c.transAxes, fontsize=9, verticalalignment='top',
              bbox=dict(boxstyle="round,pad=0.5", fc="#F8F9F9", ec="#BDC3C7", lw=1.2))

    front_patch = mpatches.Patch(color='royalblue', alpha=0.75, label='Front Layer Components')
    back_patch = mpatches.Patch(facecolor='none', edgecolor='darkgreen', linestyle='--',
                                label='Back Layer Components (Projected Outline)')
    ax_c.legend(handles=[front_patch, back_patch], loc='lower left')

    ax_c.set_xlim(-panel_width/2 - 10, panel_width/2 + 10)
    ax_c.set_ylim(-panel_height/2 - 10, panel_height/2 + 10)
    ax_c.set_aspect('equal')
    ax_c.grid(True, linestyle=':', alpha=0.4)

    # ---- Data table ----
    headers = ["Component Name", "Layer Placement", "Mass (kg)", "Dimensions (mm)", "CoG X (mm)", "CoG Y (mm)"]
    ui_table = ax_table.table(cellText=table_data, colLabels=headers, loc='center', cellLoc='center')
    ui_table.auto_set_font_size(False)
    ui_table.set_fontsize(9)
    ui_table.scale(1.0, 1.3)
    for col_idx in range(len(headers)):
        cell = ui_table[0, col_idx]
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2C3E50')

    plt.tight_layout()
    dist_path = os.path.join(MULTI_DIR, f"layout_{layout_idx}_distance_map.png")
    plt.savefig(dist_path, dpi=200, bbox_inches='tight')
    plt.close(fig_c)

    print(f"[✓] Layout {layout_idx}: saved {main_path} and {dist_path}")

# ---- Render all layouts and export combined CSV ----
all_rows = []
total_mass = masses.sum()

for layout_idx, lay in enumerate(distinct_layouts):
    cx, cy = lay["cx"], lay["cy"]
    cg_x = sum(abs_x_of_layout(cx, i) * masses[i] for i in range(n)) / total_mass
    cg_y = sum(cy[i] * masses[i] for i in range(n)) / total_mass

    render_layout(layout_idx+1, cx, cy, cg_x, cg_y)

    # Collect data for combined CSV
    abs_x_positions = np.where(is_back_side == 1, -cx, cx)
    for i in range(n):
        x_abs = abs_x_positions[i]
        y = cy[i]
        dims = f"{lengths[i]:.1f} x {widths[i]:.1f}" if element_shapes[i] == 'rectangle' else f"Ø {lengths[i]:.1f}"
        all_rows.append({
            "Layout": layout_idx + 1,
            "Layout CG Penalty": round(lay["cg"], 4),
            "Component Name": element_names[i],
            "Layer Placement": "BACK" if is_back_side[i] else "FRONT",
            "Mass (kg)": round(masses[i], 3),
            "Dimensions (mm)": dims,
            "CoG X (mm)": round(x_abs, 2),
            "CoG Y (mm)": round(y, 2),
        })

df_all = pd.DataFrame(all_rows)
csv_path = os.path.join(MULTI_DIR, "all_layouts_table.csv")
df_all.to_csv(csv_path, index=False)
print(f"[✓] Saved combined table: {csv_path}")
print(f"[✓] Done — {len(distinct_layouts)} layouts rendered in {MULTI_DIR}/")

[✓] Layout 1: saved optimized_layouts_rl/multi_layouts/layout_1_main.png and optimized_layouts_rl/multi_layouts/layout_1_distance_map.png
[✓] Layout 2: saved optimized_layouts_rl/multi_layouts/layout_2_main.png and optimized_layouts_rl/multi_layouts/layout_2_distance_map.png
[✓] Layout 3: saved optimized_layouts_rl/multi_layouts/layout_3_main.png and optimized_layouts_rl/multi_layouts/layout_3_distance_map.png
[✓] Layout 4: saved optimized_layouts_rl/multi_layouts/layout_4_main.png and optimized_layouts_rl/multi_layouts/layout_4_distance_map.png
[✓] Layout 5: saved optimized_layouts_rl/multi_layouts/layout_5_main.png and optimized_layouts_rl/multi_layouts/layout_5_distance_map.png
[✓] Saved combined table: optimized_layouts_rl/multi_layouts/all_layouts_table.csv
[✓] Done — 5 layouts rendered in optimized_layouts_rl/multi_layouts/
